# AIAA 3102 — Embedding-tied masked language modeling

Run all cells in this folder. This reproduces the clustered decoder from the bundled encoder-only initialization and static corpus masks, then trains a new head tied to its input embedding. All of the third book is reserved for model selection.


In [ ]:
from make_data import HERE, generate, digest
from tied_train import TRAIN_SHA, DEV_SHA
manifest = generate()
assert manifest["train"]["bytes"] <= 50_000_000
assert manifest["train"]["sha256"] == TRAIN_SHA and manifest["dev"]["sha256"] == DEV_SHA


In [ ]:
from train import run as fit_v1

replay_dir = HERE / "replay"
replay_dir.mkdir(exist_ok=True)
baseline_history = fit_v1(out_dir=replay_dir)
base_checkpoint = replay_dir / "base.pt"
assert digest(replay_dir / "model.pt") == baseline_history["model_sha256"]
(replay_dir / "model.pt").replace(base_checkpoint)


In [ ]:
from tied_train import run as fit_tied

tied_history = fit_tied(base_checkpoint, replay_dir, pilot=False)


In [ ]:
from shutil import copyfile
from student_model import load_model

selected = HERE / tied_history["winner_path"]
assert digest(selected) == tied_history["model_sha256"]
model = load_model(selected, "cpu")
assert sum(p.numel() for p in model.parameters()) == 7_064_891
assert model.output_embedding is model.bert.embeddings.word_embeddings.weight
assert not model.tied_only.item()
temporary = HERE / "model.tmp"
copyfile(selected, temporary)
assert digest(temporary) == tied_history["model_sha256"]
temporary.replace(HERE / "model.pt")
print(f'Saved the selected mixture checkpoint to model.pt')
